# 05 — Cruce y enriquecimiento de necropsias

**Proyecto:** Cotejo ante-mortem / post-mortem de personas desaparecidas (Querétaro, México) — MNA Integrador.

Este notebook cruza las dos tablas de necropsias del catálogo `proyecto_mna.fichas_busqueda`:

* **`registro_necropsias_2020_2026`** (la **NUEVA**): 1,246 filas, años 2006–2026. Es el **superconjunto**. Aporta amplitud de cobertura, pero sus columnas son más toscas y **no** tiene columna explícita de tatuajes.
* **`registro_necropsias_2026`** (la **VIEJA**): 58 filas. Es un **subconjunto completo** de la nueva (los 58 `#` coinciden con `No` de la nueva). Aporta **detalle forense** para esos casos: tatuajes, cicatrices, edad, municipio, genética, causa de muerte, etc.

**Estrategia:** `LEFT JOIN` nueva → vieja sobre `No = #`. Así se conservan las 1,246 filas de la nueva y se enriquecen los 58 casos compartidos con el detalle forense de la vieja. Para los 1,188 casos que solo están en la nueva, las columnas de la vieja quedan NULL y se construye un `texto_canonico_nec` a partir de ropa + pertenencias + síntesis.

La tabla resultante (`necropsias_enriquecidas`) será la fuente para cotejar contra `fichas_persona_silver` mediante Vector Search, usando campos de bloqueo (`sexo_norm`, `edad`, `municipio_hechos`) y texto canónico (`descr_tatuajes` o `texto_canonico_nec`).

In [0]:
from pyspark.sql.functions import col, when, trim, upper, concat_ws, lit, coalesce, regexp_replace

# Nombres de las dos tablas (catalog.schema.table)
TBL_NUEVA = "proyecto_mna.fichas_busqueda.registro_necropsias_2020_2026"  # 1,246 filas, superconjunto
TBL_VIEJA = "proyecto_mna.fichas_busqueda.registro_necropsias_2026"      # 58 filas, subconjunto con detalle forense

# ------------------------------------------------------------------
# Tabla NUEVA: renombrar columnas sucias (saltos de línea \n / espacio final)
# a nombres limpios. Pasamos el string EXACTO original a withColumnRenamed.
# ------------------------------------------------------------------
nueva = (spark.table(TBL_NUEVA)
    .withColumnRenamed("No", "no_nec")
    .withColumnRenamed("AÑO", "anio")
    .withColumnRenamed("FECHA DE INGRESO AL ANFITEATRO", "fecha_ingreso")
    .withColumnRenamed("HORA DE NECROPSIA Y/O ANTROPOLOGICO", "hora_necropsia")
    .withColumnRenamed("TIPO\n (QUÉ INGRESA A SEMEFO)", "tipo_ingreso")   # ¡contiene un \n!
    .withColumnRenamed("SEXO\n", "sexo_nec")                              # ¡contiene un \n!
    .withColumnRenamed("STATUS", "status")
    .withColumnRenamed("SINTESIS DE LOS HECHOS", "sintesis")
    .withColumnRenamed("ESPECIFICAR CONDICIONES EN QUE INGRESA EL CADAVER AL SEMEFO", "condiciones_ingreso")
    .withColumnRenamed("DESCRIPCION DE ROPA QUE VESTIA", "ropa")
    .withColumnRenamed("DESCRIPCION DE PERTENENCIAS QUE PERMITAN LA IDENTIFICACION ", "pertenencias")  # ¡espacio final!
)

# ------------------------------------------------------------------
# Tabla VIEJA: renombrar a nombres limpios. Las columnas de detalle
# forense se conservan para enriquecer los 58 casos compartidos.
# ------------------------------------------------------------------
vieja = (spark.table(TBL_VIEJA)
    .withColumnRenamed("#", "no_vieja")
    .withColumnRenamed("FECHA RECEPCIÓN", "fecha_recepcion")
    .withColumnRenamed("CADENA DE CUSTODIA", "cadena_custodia")
    .withColumnRenamed("MUNICIPIO DONDE OCURRIÓ EL HECHO", "municipio_hechos")
    .withColumnRenamed("EDAD (AÑOS, MESES, SEMANAS, DÍAS)", "edad")
    .withColumnRenamed("SEXO", "sexo_viejo")
    .withColumnRenamed("TIPO DE INDICIO", "tipo_indicio")
    .withColumnRenamed("CONDICIONES EN QUE INGRESA EL INDICIO", "condiciones_indicio")
    .withColumnRenamed("INGRESA COMO IDENTIFICADO / NO IDENTIFICADO", "ingreso_identificado")
    .withColumnRenamed("OCUPACIÓN", "ocupacion")
    .withColumnRenamed("CAUSA DE MUERTE", "causa_muerte")
    .withColumnRenamed("CRONODIAGNOSTICO", "cronodiagnostico")
    .withColumnRenamed("TIPO DE MUERTE", "tipo_muerte")
    .withColumnRenamed("HECHO QUE OCASIONÓ LA MUERTE", "hecho_muerte")
    .withColumnRenamed("RESEÑA DEL HECHO", "resena_hecho")
    .withColumnRenamed("QUÍMICA", "quimica")
    .withColumnRenamed("GENÉTICA", "genetica")
    .withColumnRenamed("DESCRIPCIÓN DE VESTIMENTAS", "descr_vestimenta")
    .withColumnRenamed("CICATRICES", "tiene_cicatriz")
    .withColumnRenamed("TATUAJES", "tiene_tatuajes")
    .withColumnRenamed("DESCRIPCIÓN TATUAJES", "descr_tatuajes")
    .withColumnRenamed("¿SE RECONOCE?", "se_reconoce")
    .withColumnRenamed("¿EL CUERPO INGRESA A FRIGORIFICOS? INDIQUE EN QUE GAVETA", "frigorificos_gaveta")
    .withColumnRenamed("ESTADO ACTUAL DEL INDICIO", "estado_indicio")
)

print("=== Tabla NUEVA (registro_necropsias_2020_2026) ===")
nueva.printSchema()
print("Filas:", nueva.count())

print("\n=== Tabla VIEJA (registro_necropsias_2026) ===")
vieja.printSchema()
print("Filas:", vieja.count())

In [0]:
# ------------------------------------------------------------------
# Validar el traslape entre ambas tablas.
# Clave de cruce: nueva.no_nec  ==  vieja.no_vieja
# Esperado: la vieja (58) es subconjunto COMPLETO de la nueva (1,246):
#   - traslape = 58
#   - solo_en_vieja = 0
# ------------------------------------------------------------------
join_expr = col("no_nec") == col("no_vieja")

traslape    = nueva.join(vieja, join_expr, "inner").count()
solo_nueva  = nueva.join(vieja, join_expr, "left_anti").count()
solo_vieja  = vieja.join(nueva, join_expr, "left_anti").count()

print(f"Traslape (en ambas):  {traslape}")
print(f"Solo en la NUEVA:     {solo_nueva}")
print(f"Solo en la VIEJA:     {solo_vieja}   (debe ser 0)")
print(f"Total NUEVA:          {traslape + solo_nueva}")
print(f"Total VIEJA:          {traslape + solo_vieja}")

assert solo_vieja == 0, "¡La vieja debería ser subconjunto completo de la nueva!"
assert (traslape + solo_nueva) == nueva.count(), "La suma no cuadra con el total de la nueva"
print("\n✔ Confirmado: la vieja es subconjunto completo de la nueva.")

In [0]:
# ------------------------------------------------------------------
# Construir el DataFrame enriquecido con LEFT JOIN nueva → vieja.
# Conserva TODAS las filas de la nueva (1,246) y aporta el detalle
# forense de la vieja para los 58 casos compartidos; el resto queda NULL.
# ------------------------------------------------------------------
enriquecida = nueva.join(vieja, col("no_nec") == col("no_vieja"), "left")

# --- texto_canonico_nec ---
# Concatenación limpia de ropa + pertenencias + síntesis (mayúsculas,
# sin saltos de línea, sin espacios dobles). Sirve como señal de texto
# para los casos que NO tienen tatuajes registrados en la vieja (1,188).
def _limpiar(c):
    return regexp_replace(trim(regexp_replace(upper(col(c)), r"[\r\n]+", " ")), r"\s+", " ")

texto_canonico_nec = concat_ws(" | ", _limpiar("ropa"), _limpiar("pertenencias"), _limpiar("sintesis"))
enriquecida = enriquecida.withColumn("texto_canonico_nec", texto_canonico_nec)

# --- tiene_tatuaje_bin (SI/NO normalizado) ---
# Prioridad 1: campo TATUAJES de la vieja (SI/NO).
# Prioridad 2: si la vieja no aporta, buscar 'tatuaje' (case-insensitive)
#             en los textos de la nueva (ropa, pertenencias, síntesis).
texto_busca_tatuaje = concat_ws(" ", col("ropa"), col("pertenencias"), col("sintesis"))
tiene_tatuaje_bin = (
    when(col("tiene_tatuajes") == "SI", lit("SI"))
    .when(col("tiene_tatuajes") == "NO", lit("NO"))
    .when(upper(texto_busca_tatuaje).like("%TATUAJE%"), lit("SI"))
    .otherwise(lit("NO"))
)
enriquecida = enriquecida.withColumn("tiene_tatuaje_bin", tiene_tatuaje_bin)

print("Filas enriquecidas (debe ser 1,246):", enriquecida.count())
print("\nDistribución tiene_tatuaje_bin:")
enriquecida.groupBy("tiene_tatuaje_bin").count().orderBy(col("count").desc()).show()

In [0]:
# ------------------------------------------------------------------
# Normalización de SEXO.
# sexo_nec (nueva) viene sucio: MASCULINO, Masculino, Femenino, FEMENINO,
#   FEMENINA, INDETERMINADO, NO VALORABLE, FETO, POR VALORAR, etc.
# sexo_viejo (vieja) es más limpio: HOMBRE / MUJER / INDETERMINADO.
#
# sexo_norm: usar sexo_viejo cuando sea confiable (HOMBRE/MUJER);
#           si no existe o es INDETERMINADO, normalizar sexo_nec.
# ------------------------------------------------------------------
snu = upper(trim(col("sexo_nec")))
sexo_nec_norm = (
    when(snu.rlike(r"^MASCULINO"), lit("HOMBRE"))
    .when(snu.rlike(r"^FEMEN"), lit("MUJER"))
    .otherwise(lit("INDETERMINADO"))
)

sexo_norm = coalesce(
    when(col("sexo_viejo").isin("HOMBRE", "MUJER"), col("sexo_viejo")),
    sexo_nec_norm
)
enriquecida = enriquecida.withColumn("sexo_norm", sexo_norm)

# --- Auditoría de discrepancias ---
# Filas donde sexo_viejo (confiable) difiere de la normalización de sexo_nec.
# Ya detectamos ejemplos: No=205 (nueva FEMENINO vs vieja HOMBRE),
# No=73 (nueva MASCULINO vs vieja INDETERMINADO).
discrep = enriquecida.filter(
    col("sexo_viejo").isNotNull()
    & col("sexo_viejo").isin("HOMBRE", "MUJER")
    & (col("sexo_viejo") != sexo_nec_norm)
)
print(f"Discrepancias sexo_viejo vs sexo_nec normalizado: {discrep.count()}")
discrep.select("no_nec", "anio", "sexo_nec", "sexo_viejo", "sexo_norm") \
        .orderBy("no_nec").show(50, truncate=False)

print("\nDistribución sexo_norm:")
enriquecida.groupBy("sexo_norm").count().orderBy(col("count").desc()).show()

In [0]:
# ------------------------------------------------------------------
# Guardar el resultado en Unity Catalog y verificar.
# ------------------------------------------------------------------
TBL_DESTINO = "proyecto_mna.fichas_busqueda.necropsias_enriquecidas"

# Orden de columnas final para legibilidad y uso downstream.
cols_finales = [
    "no_nec", "anio", "fecha_ingreso", "hora_necropsia", "tipo_ingreso",
    "sexo_nec", "sexo_viejo", "sexo_norm", "status",
    "sintesis", "ropa", "pertenencias", "texto_canonico_nec",
    "tiene_tatuajes", "descr_tatuajes", "tiene_tatuaje_bin",
    "tiene_cicatriz", "descr_vestimenta",
    "edad", "municipio_hechos",
    "causa_muerte", "tipo_muerte", "hecho_muerte", "resena_hecho",
    "genetica", "quimica", "ocupacion", "estado_indicio",
]

# La tabla es NUEVA: usamos saveAsTable (crea la tabla; si ya existiera de
# una ejecución previa, se omite la escritura para no destruir datos sin confirmar).
if spark.catalog.tableExists(TBL_DESTINO):
    print(f"ℹ La tabla {TBL_DESTINO} ya existe. Se omite la escritura (usa DROP manual para regenerar).")
else:
    enriquecida.select(*cols_finales).write.saveAsTable(TBL_DESTINO)
    print(f"✔ Tabla creada: {TBL_DESTINO}")

final = spark.table(TBL_DESTINO)
print(f"✔ Tabla guardada: {TBL_DESTINO}")
print(f"Filas totales (debe ser 1,246): {final.count()}")
print(f"Casos con tatuaje (tiene_tatuaje_bin='SI'): "
      f"{final.filter(col('tiene_tatuaje_bin') == 'SI').count()}")

print("\nNO IDENTIFICADO con tatuajes (muestra):")
final.filter((col("status").like("NO IDENTIFICADO%")) & (col("tiene_tatuaje_bin") == "SI")) \
     .select("no_nec", "anio", "sexo_norm", "edad", "municipio_hechos",
             "descr_tatuajes", "tiene_tatuaje_bin") \
     .show(20, truncate=False)

## Notas y próximos pasos

* La tabla `proyecto_mna.fichas_busqueda.necropsias_enriquecidas` conserva las **1,246 filas** de la nueva y enriquece los **58 casos** compartidos con el detalle forense de la vieja (tatuajes, cicatrices, edad, municipio, genética).
* **Campos de bloqueo** para Vector Search: `sexo_norm` (HOMBRE/MUJER/INDETERMINADO), `edad` (texto tipo "40-50 AÑOS" — requiere normalización a rango numérico), `municipio_hechos` (solo para los 58 enriquecidos).
* **Texto canónico** para embeddings: usar `descr_tatuajes` cuando exista (casos con tatuaje); si no, `texto_canonico_nec` (ropa + pertenencias + síntesis).
* **Discrepancias de SEXO** entre las dos tablas se auditaron arriba; la vieja se consideró más confiable cuando reporta HOMBRE/MUJER.
* **Siguiente notebook:** cotejar esta tabla contra `fichas_persona_silver` (75 fichas) usando el índice de Vector Search `texto_completo_sin_features`, con bloqueo por `sexo_norm` y post-filtrado por edad/región.